# X25 Stage 2 — update-strength sweep

Stage 1 measured delta +0.0000: single weak updates don't move sampling. This sweeps strength (steps x LR x cadence) against the LR=0 control on the same 20 frozen stories. Bar: mean paired delta >= +0.02 with gain on >=11/20 stories. Writes `x25_stage2.json`.

In [ ]:
import glob, os, shutil, subprocess, sys
work = '/kaggle/working/kalia'
if os.path.exists(work):
    shutil.rmtree(work)
r = subprocess.run(['git', 'clone', '--quiet', '--branch', 'main', 'https://github.com/subhajitlucky/kalia.git', work], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
os.chdir(work)
print('code ready')
ckpt_hits = sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True))
assert ckpt_hits, 'attach kalia-v020-ckpt'
CKPT = ckpt_hits[0]
import torch
print('gpu:', torch.cuda.is_available())

In [ ]:
import json
import tiktoken
import torch
from eval_entities import entity_report
from sample import load_model
from ttt import generate_with_ttt
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
enc = tiktoken.get_encoding('gpt2')
STORIES = json.loads(open('eval/ttt_probe_stories.json').read())
model = load_model(CKPT, DEVICE)
print('stories:', len(STORIES), '| checkpoint loaded')

In [ ]:
ARMS = [('control', 256, 1, 0.0), ('strong-steps', 256, 4, 1e-4), ('strong-lr', 256, 1, 1e-3), ('tight-chunks', 128, 2, 1e-4)]
results = []
for story in STORIES:
    prompt_ids = torch.tensor([enc.encode(story['prompt'])], dtype=torch.long)
    row = {'id': story['id']}
    for tag, chunk, steps, lr in ARMS:
        out, diag = generate_with_ttt(model, prompt_ids.to(DEVICE), max_new_tokens=1024, chunk_tokens=chunk, ttt_steps=steps, ttt_lr=lr, temperature=0.8, top_k=200, seed=7)
        continuation = enc.decode(out[0, prompt_ids.size(1):].tolist())
        report = entity_report(story['prompt'], continuation)
        row[tag] = {'retention': report['retention'], 'updates': diag['updates']}
    results.append(row)
    print(f"{story['id']}: " + ' '.join(f"{t}={row[t]['retention']:.4f}" for t, _, _, _ in ARMS), flush=True)
json.dump(results, open('/kaggle/working/x25_stage2.json', 'w'), indent=2)
print('wrote x25_stage2.json')

In [ ]:
import statistics
for tag, _, _, _ in ARMS[1:]:
    deltas = [r[tag]['retention'] - r['control']['retention'] for r in results]
    pos = sum(1 for d in deltas if d > 0)
    print(f"{tag:14s} mean delta={statistics.mean(deltas):+.4f} positive={pos}/20 {'PASS' if statistics.mean(deltas) >= 0.02 and pos >= 11 else 'null'}")
print('Bar: mean>=+0.02 with gain on >=11/20 stories.')